# Blending Regressor on California Housing Dataset:

This notebook implements **Blending** — a holdout-based variation of stacking — to predict median house values on the California Housing dataset.

### How Blending Differs from Stacking:
* **Stacking** uses $k$-fold cross-validation (`cv=10`) to generate out-of-fold predictions across the entire training set.
* **Blending** splits the training data into a **base training set** and a single **holdout validation set**. Base models train on the base set and predict on the holdout set, which serves as training data for the meta-learner.

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.pipeline import make_pipeline
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error

## 1. Create Train, Holdout Validation, and Test Splits:
Split the data into **60% base training**, **20% holdout validation** (for training the meta-learner), and **20% final test** sets.

In [2]:
X, y = fetch_california_housing(return_X_y=True)

In [3]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=42)

print("Base Train:", X_train.shape, "| Holdout Val:", X_val.shape, "| Test:", X_test.shape)

Base Train: (12384, 8) | Holdout Val: (4128, 8) | Test: (4128, 8)


## 2. Train Base Models & Generate Meta-Features:
Fit each base estimator on `X_train`, then stack their predictions on `X_val` and `X_test` column-wise to form meta-features.

In [4]:
base_models = [
    ('lr', make_pipeline(StandardScaler(), LinearRegression())),
    ('dtreg', DecisionTreeRegressor(max_depth=8, random_state=42)),
    ('gdboost', GradientBoostingRegressor(random_state=42))
]

In [5]:
val_preds = []
test_preds = []

for name, model in base_models:
    model.fit(X_train, y_train)
    val_preds.append(model.predict(X_val))
    test_preds.append(model.predict(X_test))

meta_X_val = np.column_stack(val_preds)
meta_X_test = np.column_stack(test_preds)

## 3. Train Meta-Learner & Evaluate Performance:
Fit a regularized `RidgeCV` meta-learner on the holdout validation predictions (`meta_X_val`) and evaluate final blended predictions on `meta_X_test`.

In [6]:
meta_model = RidgeCV()
meta_model.fit(meta_X_val, y_val)

,"alphas alphas: array-like of shape (n_alphas,), default=(0.1, 1.0, 10.0)Array of alpha values to try.Regularization strength; must be a positive float. Regularizationimproves the conditioning of the problem and reduces the variance ofthe estimates. Larger values specify stronger regularization.Alpha corresponds to ``1 / (2C)`` in other linear models such as:class:`~sklearn.linear_model.LogisticRegression` or:class:`~sklearn.svm.LinearSVC`.If using Leave-One-Out cross-validation, alphas must be strictly positive.For an example on how regularization strength affects the model coefficients,see :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`.","(0.1, ...)"
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"scoring scoring: str, callable, default=NoneThe scoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: negative :ref:`mean squared error <mean_squared_error>` if cv is None (i.e. when using leave-one-out cross-validation), or :ref:`coefficient of determination <r2_score>` (:math:`R^2`) otherwise.",None
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the efficient Leave-One-Out cross-validation- integer, to specify the number of folds,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used, else,:class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here.",None
,"gcv_mode gcv_mode: {'auto', 'svd', 'eigen'}, default='auto'Flag indicating which strategy to use when performingLeave-One-Out Cross-Validation. Options are:: 'auto' : same as 'eigen' 'svd' : use singular value decomposition of X when X is dense, fallback to 'eigen' when X is sparse 'eigen' : use eigendecomposition of X X' when n_samples <= n_features or X' X when n_features < n_samplesThe 'auto' mode is the default and is intended to pick the cheaperoption depending on the shape and sparsity of the training data.",None
,"store_cv_results store_cv_results: bool, default=FalseFlag indicating if the cross-validation values corresponding toeach alpha should be stored in the ``cv_results_`` attribute (seebelow). This flag is only compatible with ``cv=None`` (i.e. usingLeave-One-Out Cross-Validation)... versionchanged:: 1.5 Parameter name changed from `store_cv_values` to `store_cv_results`.",False
,"alpha_per_target alpha_per_target: bool, default=FalseFlag indicating whether to optimize the alpha value (picked from the`alphas` parameter list) for each target separately (for multi-outputsettings: multiple prediction targets). When set to `True`, afterfitting, the `alpha_` attribute will contain a value for each target.When set to `False`, a single alpha is used for all targets.This flag is only compatible with ``cv=None`` (i.e. usingLeave-One-Out Cross-Validation)... versionadded:: 0.24",False
Name,Type,Value
"alpha_ alpha_: float or ndarray of shape (n_targets,)Estimated regularization parameter, or, if ``alpha_per_target=True``,the estimated regularization parameter for each target.",float,1
"best_score_ best_score_: float or ndarray of shape (n_targets,)Score of base estimator with best alpha, or, if``alpha_per_target=True``, a score for each target... versionadded:: 0.23",float64,-0.2804
"coef_ coef_: ndarray of shape (n_features) or (n_targets, n_features)Weight vector(s).","ndarray[float64](3,)","[-0.05, 0.08, 1.01]"


In [7]:
y_pred = meta_model.predict(meta_X_test)

print("R2 Score:", r2_score(y_test, y_pred))
print("RMSE:", root_mean_squared_error(y_test, y_pred))

R2 Score: 0.7807747006757311
RMSE: 0.5359802239747943
